In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
from models.aft import generate_aft_data
from matplotlib.ticker import ScalarFormatter

# 寻找所有的 aft 结果文件
files = glob.glob('aft/*.json')
if not files:
    print("在 aft/ 文件夹下没有找到 JSON 文件。请先运行 aft_run_parallel.ipynb")

for filename in files:
    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
    res = data['results']
    params = data['parameters']
    N = len(res)
    
    print(f"================ 文件: {os.path.basename(filename)} ================")
    md_table = f"### {params.get('noise_type', 'normal')} 噪声下的性能对比 (AFT, N={N})\n"
    md_table += "| Method | RMSE | MAE | F1 | Prec | Rec | Pairwise_Acc | Time (s) |\n"
    md_table += "|---|---|---|---|---|---|---|---|\n"
    
    # 尽可能展示所有出现的算法
    all_methods = ['Global', 'Local', 'Avg', 'D-subGD', 'D-ProxGD', 'U-ADMM']
    all_hists = {}
    final_rmses = {}
    
    for method in all_methods:
        if method in res[0]:
            rmses = [r[method]['RMSE'] for r in res if 'RMSE' in r[method]]
            maes = [r[method]['MAE'] for r in res if 'MAE' in r[method]]
            accs = [r[method].get('Pairwise_Correlation', 0.0) for r in res]
            times = [r[method].get('Time', 0.0) for r in res]
            
            # 补算 F1, Prec, Rec
            if 'F1_Score' not in res[0][method] and 'theta_hat' in res[0][method]:
                from utils.eval_utils import calculate_metrics
                f1s = []
                precs = []
                recs = []
                for r in res:
                    if method in r and 'theta_hat' in r[method]:
                        theta_hat = np.array(r[method]['theta_hat'])
                        theta_true_tmp = np.zeros_like(theta_hat)
                        theta_true_tmp[:params.get('p_prime', 5)] = 1.0
                        m_dict = calculate_metrics(theta_true_tmp, theta_hat)
                        f1s.append(m_dict['F1_Score'])
                        precs.append(m_dict['Precision'])
                        recs.append(m_dict['Recall'])
            else:
                f1s = [r[method].get('F1_Score', 0.0) for r in res]
                precs = [r[method].get('Precision', 0.0) for r in res]
                recs = [r[method].get('Recall', 0.0) for r in res]
            
            if len(rmses) > 0:
                mean_rmse = np.mean(rmses)
                final_rmses[method] = mean_rmse
                md_table += f"| {method} | {mean_rmse:.4f} | {np.mean(maes):.4f} | {np.mean(f1s):.4f} | {np.mean(precs):.4f} | {np.mean(recs):.4f} | {np.mean(accs):.2%} | {np.mean(times):.2f} |\n"
                
                if 'hist_rmse' in res[0][method]:
                    all_hists[method] = np.mean([r[method]['hist_rmse'] for r in res], axis=0)
    
    print(md_table)
    
    
    # ============================================================
    # 1. 散点图绘制逻辑
    # ============================================================
    print("\n[绘图] 正在生成并排预测散点图...")
    test_seed = params.get('rng_seed', 42) + 999 
    d_test = generate_aft_data(
        m=params['m'], n=params['n'], p_prime=params.get('p_prime', 5), 
        p=params['p'], pc=params['pc'], cens_target=params.get('cens_target', 0.25), noise_type=params['noise_type'], 
        rng_seed=test_seed, noise_scale=params.get('noise_scale', 1.0)
    )
    X_test = d_test['X']
    theta_true = d_test['theta_true']
    true_scores = (X_test @ theta_true).flatten()
    res_one = res[0] 
    
    plot_groups = [(['U-ADMM', 'Global'], 'U-ADMM & Global'), (['Local', 'Avg'], 'Local & Avg')]
    colors = {'U-ADMM': '#3498DB', 'Global': '#FFA500', 'Local': '#2ECC71', 'Avg': '#E74C3C'}
    dark_colors = {'U-ADMM': '#1F618D', 'Global': '#B9770E', 'Local': '#1E8449', 'Avg': '#922B21'}
    
    # 寻找全局统一的上下限
    score_min, score_max = true_scores.min(), true_scores.max()
    for m_key in ['U-ADMM', 'Global', 'Local', 'Avg']:
        if m_key in res_one and 'theta_hat' in res_one[m_key]:
            p_scores = (X_test @ np.array(res_one[m_key]['theta_hat']).reshape(-1, 1)).flatten()
            score_min, score_max = min(score_min, p_scores.min()), max(score_max, p_scores.max())
            
    pad = (score_max - score_min) * 0.05
    limit_range = [score_min - pad, score_max + pad]

    # 画1行2列的图
    fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
    
    for i, (methods, title) in enumerate(plot_groups):
        ax = axes[i]
        
        for m_key in methods:
            if m_key in res_one and 'theta_hat' in res_one[m_key]:
                theta_hat = np.array(res_one[m_key]['theta_hat']).reshape(-1, 1)
                pred_scores = (X_test @ theta_hat).flatten()
                ax.scatter(true_scores, pred_scores, s=15, alpha=0.5, color=colors.get(m_key), label=m_key)
                
                # 绘制拟合直线
                if len(true_scores) > 1:
                    m_fit, b_fit = np.polyfit(true_scores, pred_scores, 1)
                    fit_x = np.array(limit_range)
                    fit_y = m_fit * fit_x + b_fit
                    ax.plot(fit_x, fit_y, '-', color=dark_colors.get(m_key, colors.get(m_key)), linewidth=2)
                
        ax.plot(limit_range, limit_range, 'k--', alpha=0.5, label='Ideal y=x')
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlim(limit_range)
        ax.set_ylim(limit_range)
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('True Latent Scores')
        if i == 0: ax.set_ylabel('Predicted Scores')
        ax.grid(True, alpha=0.2)
        ax.legend(loc='upper left')
        
    plt.tight_layout()
    plt.show()

    # ============================================================
    # 2. 合并收敛曲线图
    # ============================================================
    if len(all_hists) > 0 or len(final_rmses) > 0:
        print("\n[绘图] 正在生成合并收敛轨迹图 (包含所有算法与后备对比线)...")
        plt.figure(figsize=(9, 5.5))
        
        style_map = {
            'U-ADMM': ('-o', '#3498DB'), 
            'Global': ('-', '#FFA500'),
            'D-subGD': ('--', '#E67E22'),
            'D-ProxGD': ('--', '#9B59B6')
        }
        
        # 寻找全局的最大步数
        max_steps = 0
        if 'U-ADMM' in all_hists:
            max_steps = max(max_steps, (len(all_hists['U-ADMM']) - 1) * params.get('W_inner', 5))
        for m_key, hist in all_hists.items():
            if m_key != 'U-ADMM':
                max_steps = max(max_steps, len(hist) - 1)
        if max_steps == 0:
            max_steps = 100
        
        iterative_methods = ['U-ADMM', 'Global', 'D-subGD', 'D-ProxGD']
        
        for m_key in iterative_methods:
            if m_key in final_rmses:
                color = style_map.get(m_key, ('-', 'k'))[1]
                marker = style_map.get(m_key, ('-', 'k'))[0]
                
                if m_key in all_hists:
                    hist = all_hists[m_key]
                    if m_key == 'U-ADMM':
                        steps = np.arange(len(hist)) * params.get('W_inner', 5)
                    else:
                        steps = np.arange(len(hist))
                    
                    plt.plot(steps, hist, marker, color=color, label=m_key, markersize=4, alpha=0.8)
                else:
                    plt.hlines(final_rmses[m_key], 0, max_steps, colors=color, linestyles=':', 
                               label=f"{m_key} (RMSE={final_rmses[m_key]:.4f})")
            
        if 'Avg' in final_rmses:
            plt.hlines(final_rmses['Avg'], 0, max_steps, colors='#E74C3C', linestyles='--', 
                       label=f"Avg (RMSE={final_rmses['Avg']:.4f})")
        if 'Local' in final_rmses:
            plt.hlines(final_rmses['Local'], 0, max_steps, colors='#2ECC71', linestyles='-.', 
                       label=f"Local (RMSE={final_rmses['Local']:.4f})")
            
        plt.yscale('linear')
        ax = plt.gca()
        ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))
        ax.ticklabel_format(style='plain', axis='y')
        
        plt.xlabel('Total Iterations (t)')
        plt.ylabel('RMSE')
        plt.title(f"Convergence Comparison (AFT, noise={params['noise_type']})")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, which="both", ls="-", alpha=0.2)
        plt.tight_layout()
        plt.show()

    # ============================================================
    # 3. RMSE 箱线图
    # ============================================================
    import seaborn as sns
    print("\n[绘图] 正在生成 RMSE 稳定性箱线图...")
    rmse_data = {}
    for method in all_methods:
        method_rmses = [r[method]['RMSE'] for r in res if method in r and 'RMSE' in r[method]]
        if method_rmses:
            rmse_data[method] = method_rmses

    if rmse_data:
        labels, data_box = [*zip(*rmse_data.items())] 
        plt.figure(figsize=(10, 6))
        bplot = plt.boxplot(data_box, labels=labels, patch_artist=True, showmeans=True,
                            medianprops={'color': 'black', 'linewidth': 2},
                            meanprops={'marker':'o', 'markerfacecolor':'white', 'markeredgecolor':'black'})

        box_colors = ['#FFA500', '#2ECC71', '#E74C3C', '#9B59B6', '#3498DB', '#1ABC9C']
        for patch, color in zip(bplot['boxes'], box_colors[:len(bplot['boxes'])]):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        plt.ylabel('RMSE', fontsize=12)
        plt.title(f"RMSE Distribution over {len(res)} Runs (Noise: {params.get('noise_type', 'normal')})", fontsize=14)
        plt.grid(axis='y', linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.show()

    # ============================================================
    # 4. 系数对比图 (展示单次Run的估计结果)
    # ============================================================
    print("\n[绘图] 正在生成系数估计对比图...")
    plt.figure(figsize=(10, 5))
    plt.plot(theta_true, marker='o', markersize=6, linestyle='None', label='True Coefficients', alpha=0.5, color='gray')
    
    alg_styles = {
        'U-ADMM': ('x', 8, '#4E93D9', 1.0),
        'Global': ('+', 8, 'green', 1.0),
        'D-subGD': ('x', 8, '#EE1C25', 1.0),
        'D-ProxGD': ('D', 6, '#9B59B6', 0.9),
        'Local': ('v', 6, '#2ECC71', 0.8),
        'Avg': ('^', 6, '#E74C3C', 0.8)
    }
    
    for m_key in ['U-ADMM', 'Global', 'D-ProxGD', 'D-subGD', 'Avg', 'Local']:
        if m_key in res_one and 'theta_hat' in res_one[m_key]:
            theta_hat = np.array(res_one[m_key]['theta_hat']).flatten()
            marker, ms, color, alpha = alg_styles.get(m_key, ('o', 5, 'black', 1.0))
            plt.plot(theta_hat, marker=marker, markersize=ms, color=color, linestyle='None', alpha=alpha, label=m_key)
            
    plt.axhline(0, color='black', alpha=0.2, linestyle='--')
    plt.xlabel('Coefficient Index')
    plt.ylabel('Coefficient Value')
    plt.title(f"Coefficient Comparison: True vs All Methods (Noise: {params.get('noise_type', 'normal')})")
    plt.legend(frameon=True, loc='upper right', bbox_to_anchor=(1.25, 1))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
